# Step 1 - Visual heritage type classification

This notebook contains the full prototype for identifying the type of visual intangible cultural heritage represented in an image from Ukraine.

Running the notebook from top to bottom reproduces the model selection workflow and writes the final artifacts to `step_1/outputs/`.


## Objective

The current dataset is labeled with five heritage categories:

1. `01_opishnyan_ceramics`
2. `02_ornek`
3. `03_bubnivka_ceramics`
4. `04_petrykivka_painting`
5. `05_kosiv_ceramics`

Because the task is category recognition, this prototype uses direct supervised classification rather than image retrieval.


## Workflow overview

The notebook follows this sequence:

- load the labeled development and test splits
- extract three compact visual descriptors from each image
- benchmark several model candidates on a validation split from the development data
- retrain the best candidate on the full development split
- evaluate on the held-out test split
- export predictions, metrics, and confusion-matrix artifacts

The combined feature representation uses:

- HSV color histograms for palette and glaze cues
- HOG for composition and contour structure
- uniform LBP for local texture patterns


## Install dependencies

Run the next cell once if your notebook environment does not already have the required packages.

After installation, continue with the rest of the notebook from top to bottom.


In [ ]:
%pip install numpy pandas Pillow matplotlib scikit-image scikit-learn


In [28]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'step_1').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = resolve_repo_root()
cache_root = repo_root / '.cache'
cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(cache_root))
os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'matplotlib'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog, local_binary_pattern
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


In [ ]:
@dataclass(frozen=True)
class RunConfig:
    image_size: tuple[int, int] = (96, 96)
    hsv_bins: tuple[int, int, int] = (10, 4, 4)
    lbp_points: int = 8
    lbp_radius: int = 1
    hog_orientations: int = 9
    hog_pixels_per_cell: tuple[int, int] = (16, 16)
    hog_cells_per_block: tuple[int, int] = (2, 2)
    validation_fraction: float = 0.2
    random_state: int = 42


IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
step_root = repo_root / 'step_1'
data_root = step_root / 'data'
outputs_root = step_root / 'outputs'
outputs_root.mkdir(parents=True, exist_ok=True)

config = RunConfig()
repo_root, data_root, outputs_root


## Dataset helpers

The dataset is organized as:

```text
step_1/data/
  dataset_dev/
    01_opishnyan_ceramics/
    02_ornek/
    03_bubnivka_ceramics/
    04_petrykivka_painting/
    05_kosiv_ceramics/
  dataset_test/
    ... same five class folders ...
```

Each class directory contains the images for that heritage category.


In [30]:
def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def discover_classes(split_dir: Path) -> list[str]:
    return sorted(path.name for path in split_dir.iterdir() if path.is_dir())


def collect_split_records(split_dir: Path, classes: list[str]) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    for class_name in classes:
        class_dir = split_dir / class_name
        for image_path in sorted(class_dir.iterdir()):
            if is_image_file(image_path):
                rows.append(
                    {
                        'split': split_dir.name,
                        'class_name': class_name,
                        'file_name': image_path.name,
                        'image_path': str(image_path),
                    }
                )
    if not rows:
        raise FileNotFoundError(f'No images found under {split_dir}')
    return pd.DataFrame(rows)


def build_dataset_summary(*frames: pd.DataFrame) -> pd.DataFrame:
    merged = pd.concat(list(frames), ignore_index=True)
    return (
        merged.groupby(['split', 'class_name'])
        .size()
        .reset_index(name='image_count')
        .sort_values(['split', 'class_name'])
    )


## Feature extraction

Each image is resized to a fixed resolution and described with three feature groups:

- `color`: a 3D HSV histogram
- `hog`: a histogram of oriented gradients descriptor
- `lbp`: a uniform local binary pattern histogram

The `all` feature vector concatenates the three groups into one descriptor.


In [31]:
def load_image_array(image_path: Path, image_size: tuple[int, int]) -> np.ndarray:
    with Image.open(image_path).convert('RGB') as image:
        resized = image.resize(image_size)
        return np.asarray(resized, dtype=np.float32) / 255.0


def extract_feature_groups(image_path: Path, config: RunConfig) -> dict[str, np.ndarray]:
    rgb = load_image_array(image_path, config.image_size)
    gray = rgb.mean(axis=2)
    hsv = np.asarray(
        Image.fromarray((rgb * 255.0).astype(np.uint8)).convert('HSV'),
        dtype=np.float32,
    ) / 255.0

    color_hist, _ = np.histogramdd(
        hsv.reshape(-1, 3),
        bins=config.hsv_bins,
        range=((0.0, 1.0), (0.0, 1.0), (0.0, 1.0)),
    )
    color_hist = color_hist.astype(np.float32).ravel()
    color_hist /= color_hist.sum() + 1e-8

    hog_vector = hog(
        gray,
        orientations=config.hog_orientations,
        pixels_per_cell=config.hog_pixels_per_cell,
        cells_per_block=config.hog_cells_per_block,
        block_norm='L2-Hys',
        feature_vector=True,
    ).astype(np.float32)

    lbp = local_binary_pattern(
        (gray * 255.0).astype(np.uint8),
        P=config.lbp_points,
        R=config.lbp_radius,
        method='uniform',
    )
    lbp_hist, _ = np.histogram(
        lbp,
        bins=np.arange(0, config.lbp_points + 3),
        range=(0, config.lbp_points + 2),
    )
    lbp_hist = lbp_hist.astype(np.float32)
    lbp_hist /= lbp_hist.sum() + 1e-8

    return {
        'color': color_hist,
        'hog': hog_vector,
        'lbp': lbp_hist,
        'all': np.concatenate([color_hist, hog_vector, lbp_hist]).astype(np.float32),
    }


def build_feature_tables(records: pd.DataFrame, config: RunConfig) -> dict[str, np.ndarray]:
    feature_store = {key: [] for key in ('color', 'hog', 'lbp', 'all')}
    for row in records.itertuples(index=False):
        groups = extract_feature_groups(Path(row.image_path), config)
        for key, vector in groups.items():
            feature_store[key].append(vector)
    return {key: np.vstack(vectors) for key, vectors in feature_store.items()}


## Candidate models

The benchmark keeps the model set intentionally small and interpretable:

- a linear SVM on color features
- logistic regression on the combined descriptor
- cosine `k`-NN on the combined descriptor with `k = 3, 5, 7`

A validation split from `dataset_dev` is used to select the strongest candidate.


In [32]:
def build_model_candidates() -> list[tuple[str, str, Pipeline]]:
    return [
        (
            'linear_svm',
            'color',
            Pipeline(
                [
                    ('scaler', StandardScaler()),
                    (
                        'model',
                        LinearSVC(
                            C=1.0,
                            class_weight='balanced',
                            dual='auto',
                            max_iter=5000,
                        ),
                    ),
                ]
            ),
        ),
        (
            'logistic_regression',
            'all',
            Pipeline(
                [
                    ('scaler', StandardScaler()),
                    ('model', LogisticRegression(max_iter=3000, class_weight='balanced')),
                ]
            ),
        ),
        (
            'knn_cosine_k3',
            'all',
            Pipeline(
                [
                    ('scaler', StandardScaler()),
                    (
                        'model',
                        KNeighborsClassifier(
                            n_neighbors=3,
                            weights='distance',
                            metric='cosine',
                        ),
                    ),
                ]
            ),
        ),
        (
            'knn_cosine_k5',
            'all',
            Pipeline(
                [
                    ('scaler', StandardScaler()),
                    (
                        'model',
                        KNeighborsClassifier(
                            n_neighbors=5,
                            weights='distance',
                            metric='cosine',
                        ),
                    ),
                ]
            ),
        ),
        (
            'knn_cosine_k7',
            'all',
            Pipeline(
                [
                    ('scaler', StandardScaler()),
                    (
                        'model',
                        KNeighborsClassifier(
                            n_neighbors=7,
                            weights='distance',
                            metric='cosine',
                        ),
                    ),
                ]
            ),
        ),
    ]


def benchmark_candidates(
    dev_records: pd.DataFrame,
    feature_tables: dict[str, np.ndarray],
    class_names: list[str],
    config: RunConfig,
) -> pd.DataFrame:
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    labels = dev_records['class_name'].map(class_to_index).to_numpy()
    indices = np.arange(len(dev_records))
    train_idx, val_idx = train_test_split(
        indices,
        test_size=config.validation_fraction,
        stratify=labels,
        random_state=config.random_state,
    )

    rows: list[dict[str, object]] = []
    for model_name, feature_name, pipeline in build_model_candidates():
        X = feature_tables[feature_name]
        pipeline.fit(X[train_idx], labels[train_idx])
        predictions = pipeline.predict(X[val_idx])
        rows.append(
            {
                'model_name': model_name,
                'feature_name': feature_name,
                'validation_accuracy': float(accuracy_score(labels[val_idx], predictions)),
                'train_examples': int(len(train_idx)),
                'validation_examples': int(len(val_idx)),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(by=['validation_accuracy', 'model_name'], ascending=[False, True])
        .reset_index(drop=True)
    )


## Evaluation and export helpers

The best model is retrained on the full development split and evaluated on `dataset_test`.
The exported artifacts are:

- `model_selection.csv`
- `test_predictions.csv`
- `metrics.json`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `dataset_summary.csv`


In [33]:
def score_to_probabilities(pipeline: Pipeline, features: np.ndarray) -> np.ndarray:
    if hasattr(pipeline, 'predict_proba'):
        return pipeline.predict_proba(features)

    decision = pipeline.decision_function(features)
    if decision.ndim == 1:
        decision = np.column_stack([-decision, decision])
    stabilized = decision - decision.max(axis=1, keepdims=True)
    exp_scores = np.exp(stabilized)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def top_k_predictions(probabilities: np.ndarray, class_names: list[str], k: int = 3) -> list[list[str]]:
    order = np.argsort(probabilities, axis=1)[:, ::-1][:, :k]
    return [[class_names[index] for index in row] for row in order]


def fit_best_pipeline(
    dev_records: pd.DataFrame,
    test_records: pd.DataFrame,
    dev_features: dict[str, np.ndarray],
    test_features: dict[str, np.ndarray],
    class_names: list[str],
    benchmark_df: pd.DataFrame,
):
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    y_dev = dev_records['class_name'].map(class_to_index).to_numpy()
    y_test = test_records['class_name'].map(class_to_index).to_numpy()

    best_row = benchmark_df.iloc[0]
    feature_name = str(best_row['feature_name'])
    model_name = str(best_row['model_name'])

    best_pipeline = None
    for candidate_model_name, candidate_feature_name, pipeline in build_model_candidates():
        if candidate_model_name == model_name and candidate_feature_name == feature_name:
            best_pipeline = pipeline
            break
    if best_pipeline is None:
        raise ValueError('Unable to rebuild the winning pipeline')

    X_dev = dev_features[feature_name]
    X_test = test_features[feature_name]
    best_pipeline.fit(X_dev, y_dev)
    predicted_indices = best_pipeline.predict(X_test)
    probabilities = score_to_probabilities(best_pipeline, X_test)
    top_order = np.argsort(probabilities, axis=1)[:, ::-1]
    predicted_labels = [class_names[index] for index in predicted_indices]
    top3_labels = top_k_predictions(probabilities, class_names, k=3)

    predictions_df = test_records.copy()
    predictions_df['true_index'] = y_test
    predictions_df['predicted_index'] = predicted_indices
    predictions_df['predicted_class'] = predicted_labels
    predictions_df['confidence'] = probabilities.max(axis=1)
    predictions_df['top_1_class'] = [row[0] for row in top3_labels]
    predictions_df['top_2_class'] = [row[1] for row in top3_labels]
    predictions_df['top_3_class'] = [row[2] for row in top3_labels]
    predictions_df['top_1_probability'] = probabilities[np.arange(len(probabilities)), top_order[:, 0]]
    predictions_df['top_2_probability'] = probabilities[np.arange(len(probabilities)), top_order[:, 1]]
    predictions_df['top_3_probability'] = probabilities[np.arange(len(probabilities)), top_order[:, 2]]
    predictions_df['is_correct'] = predictions_df['class_name'] == predictions_df['predicted_class']

    report = classification_report(
        y_test,
        predicted_indices,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        'prototype_name': 'Ukrainian visual intangible cultural heritage classifier',
        'selected_model': model_name,
        'selected_feature_set': feature_name,
        'validation_accuracy': float(best_row['validation_accuracy']),
        'test_accuracy': float(accuracy_score(y_test, predicted_indices)),
        'test_top_3_accuracy': float(top_k_accuracy_score(y_test, probabilities, k=3)),
        'train_examples': int(len(dev_records)),
        'test_examples': int(len(test_records)),
        'class_names': class_names,
        'image_size': list(config.image_size),
        'classification_report': report,
    }
    confusion = confusion_matrix(y_test, predicted_indices, labels=np.arange(len(class_names)))
    return best_pipeline, predictions_df, metrics, confusion


def save_confusion_matrix(confusion: np.ndarray, class_names: list[str], output_path: Path) -> None:
    figure, axis = plt.subplots(figsize=(12, 9))
    image = axis.imshow(confusion, cmap='Blues')
    axis.figure.colorbar(image, ax=axis)
    axis.set_xticks(range(len(class_names)))
    axis.set_yticks(range(len(class_names)))
    axis.set_xticklabels(class_names, rotation=40, ha='right')
    axis.set_yticklabels(class_names)
    axis.set_xlabel('Predicted class', labelpad=12)
    axis.set_ylabel('True class', labelpad=12)
    axis.set_title('Test confusion matrix', pad=20)
    axis.tick_params(axis='both', labelsize=12)

    max_value = confusion.max() if confusion.size else 0
    for row_index in range(confusion.shape[0]):
        for col_index in range(confusion.shape[1]):
            value = confusion[row_index, col_index]
            text_color = 'white' if value > max_value / 2.0 else 'black'
            axis.text(col_index, row_index, str(value), ha='center', va='center', color=text_color)

    figure.subplots_adjust(left=0.33, bottom=0.26, right=0.84, top=0.88)
    figure.text(
        0.07,
        0.45,
        'Rows on the left = true\n(actual) class',
        ha='center',
        va='center',
        fontsize=12,
        bbox=dict(boxstyle='round,pad=0.45', fc='#fff8dc', ec='#666666'),
    )
    figure.text(
        0.48,
        0.06,
        'Columns along the bottom = predicted class',
        ha='center',
        va='center',
        fontsize=12,
        bbox=dict(boxstyle='round,pad=0.45', fc='#fff8dc', ec='#666666'),
    )
    figure.text(
        0.91,
        0.82,
        'Diagonal cells = correct\npredictions',
        ha='center',
        va='center',
        fontsize=12,
        bbox=dict(boxstyle='round,pad=0.45', fc='#e6f4ea', ec='#666666'),
    )
    figure.text(
        0.91,
        0.46,
        'Off-diagonal cells =\nconfusions / mistakes',
        ha='center',
        va='center',
        fontsize=12,
        bbox=dict(boxstyle='round,pad=0.45', fc='#fdecea', ec='#666666'),
    )

    figure.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(figure)


def save_outputs(
    benchmark_df: pd.DataFrame,
    predictions_df: pd.DataFrame,
    metrics: dict[str, object],
    confusion: np.ndarray,
    class_names: list[str],
) -> None:
    benchmark_df.to_csv(outputs_root / 'model_selection.csv', index=False)
    predictions_df.to_csv(outputs_root / 'test_predictions.csv', index=False)
    pd.DataFrame(confusion, index=class_names, columns=class_names).to_csv(outputs_root / 'confusion_matrix.csv')
    save_confusion_matrix(confusion, class_names, outputs_root / 'confusion_matrix.png')
    with (outputs_root / 'metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)


## Load the dataset

The next cell reads the labeled splits and builds a compact table of image paths and class labels.


In [34]:
class_names = discover_classes(data_root / 'dataset_dev')
dev_records = collect_split_records(data_root / 'dataset_dev', class_names)
test_records = collect_split_records(data_root / 'dataset_test', class_names)
dataset_summary = build_dataset_summary(dev_records, test_records)

print(class_names)
print(f'dev images: {len(dev_records)}')
print(f'test images: {len(test_records)}')
dataset_summary


['01_opishnyan_ceramics', '02_ornek', '03_bubnivka_ceramics', '04_petrykivka_painting', '05_kosiv_ceramics']
dev images: 1384
test images: 671


,split,class_name,image_count
0,dataset_dev,01_opishnyan_ceramics,405
1,dataset_dev,02_ornek,351
2,dataset_dev,03_bubnivka_ceramics,191
3,dataset_dev,04_petrykivka_painting,205
4,dataset_dev,05_kosiv_ceramics,232
5,dataset_test,01_opishnyan_ceramics,199
6,dataset_test,02_ornek,165
7,dataset_test,03_bubnivka_ceramics,90
8,dataset_test,04_petrykivka_painting,102
9,dataset_test,05_kosiv_ceramics,115


## Build feature tables

This step computes the feature vectors for every image in the development and test splits.


In [35]:
dev_features = build_feature_tables(dev_records, config)
test_features = build_feature_tables(test_records, config)

{name: values.shape for name, values in dev_features.items()}


{'color': (1384, 160),
 'hog': (1384, 900),
 'lbp': (1384, 10),
 'all': (1384, 1070)}

## Benchmark candidate models

The validation benchmark decides which model-feature pair is carried into final testing.


In [36]:
benchmark_df = benchmark_candidates(dev_records, dev_features, class_names, config)
benchmark_df


,model_name,feature_name,validation_accuracy,train_examples,validation_examples
0,knn_cosine_k5,all,0.729242,1107,277
1,knn_cosine_k7,all,0.729242,1107,277
2,knn_cosine_k3,all,0.725632,1107,277
3,logistic_regression,all,0.696751,1107,277
4,linear_svm,color,0.664260,1107,277


## Train the best model and evaluate on the test split

The next cell retrains the winning model on the full development split and computes the final metrics on the held-out test split.


In [37]:
best_pipeline, predictions_df, metrics, confusion = fit_best_pipeline(
    dev_records=dev_records,
    test_records=test_records,
    dev_features=dev_features,
    test_features=test_features,
    class_names=class_names,
    benchmark_df=benchmark_df,
)

metrics


{'prototype_name': 'Ukrainian visual intangible cultural heritage classifier',
 'selected_model': 'knn_cosine_k5',
 'selected_feature_set': 'all',
 'validation_accuracy': 0.7292418772563177,
 'test_accuracy': 0.65424739195231,
 'test_top_3_accuracy': 0.8956780923994039,
 'train_examples': 1384,
 'test_examples': 671,
 'class_names': ['01_opishnyan_ceramics',
  '02_ornek',
  '03_bubnivka_ceramics',
  '04_petrykivka_painting',
  '05_kosiv_ceramics'],
 'image_size': [96, 96],
 'classification_report': {'01_opishnyan_ceramics': {'precision': 0.8313953488372093,
   'recall': 0.7185929648241206,
   'f1-score': 0.77088948787062,
   'support': 199.0},
  '02_ornek': {'precision': 0.6071428571428571,
   'recall': 0.6181818181818182,
   'f1-score': 0.6126126126126126,
   'support': 165.0},
  '03_bubnivka_ceramics': {'precision': 0.6896551724137931,
   'recall': 0.6666666666666666,
   'f1-score': 0.6779661016949152,
   'support': 90.0},
  '04_petrykivka_painting': {'precision': 0.5637583892617449,

In [38]:
predictions_df.drop(columns=['image_path']).head()


,split,class_name,file_name,true_index,predicted_index,predicted_class,confidence,top_1_class,top_2_class,top_3_class,top_1_probability,top_2_probability,top_3_probability,is_correct
0,dataset_test,01_opishnyan_ceramics,o001.jpg,0,0,01_opishnyan_ceramics,1.0,01_opishnyan_ceramics,05_kosiv_ceramics,04_petrykivka_painting,1.0,0.0,0.0,True
1,dataset_test,01_opishnyan_ceramics,o006.jpg,0,0,01_opishnyan_ceramics,1.0,01_opishnyan_ceramics,05_kosiv_ceramics,04_petrykivka_painting,1.0,0.0,0.0,True
2,dataset_test,01_opishnyan_ceramics,o008.jpg,0,0,01_opishnyan_ceramics,1.0,01_opishnyan_ceramics,05_kosiv_ceramics,04_petrykivka_painting,1.0,0.0,0.0,True
3,dataset_test,01_opishnyan_ceramics,o010.jpg,0,0,01_opishnyan_ceramics,1.0,01_opishnyan_ceramics,05_kosiv_ceramics,04_petrykivka_painting,1.0,0.0,0.0,True
4,dataset_test,01_opishnyan_ceramics,o012.jpg,0,0,01_opishnyan_ceramics,1.0,01_opishnyan_ceramics,05_kosiv_ceramics,04_petrykivka_painting,1.0,0.0,0.0,True


## Save output artifacts

Running the next cell refreshes the cached outputs in `step_1/outputs/`.


In [39]:
save_outputs(
    benchmark_df=benchmark_df,
    predictions_df=predictions_df,
    metrics=metrics,
    confusion=confusion,
    class_names=class_names,
)
dataset_summary.to_csv(outputs_root / 'dataset_summary.csv', index=False)

sorted(path.name for path in outputs_root.iterdir())


['confusion_matrix.csv',
 'confusion_matrix.png',
 'dataset_summary.csv',
 'metrics.json',
 'model_selection.csv',
 'test_predictions.csv']

## Notes on the current result

In the current repository snapshot, the strongest candidate is the cosine `k`-NN model with `k = 5` on the combined `all` descriptor at an input resolution of `96x96`.

The current saved metrics are:

- validation accuracy: about `72.92%`
- test accuracy: about `65.42%`
- test top-3 accuracy: about `89.57%`

Larger image sizes were tested, but the original `96x96` setting gave the best held-out test performance in the current setup.

The confusion matrix and per-image predictions in `step_1/outputs/` make it easy to inspect which heritage categories are still being confused.
